trainer.use_v1 是一个布尔开关，决定 verl 用哪一套 PPO/GRPO 训练主流程（trainer 实现）：
- use_v1=True → 走新的 V1 trainer（TaskRunnerV1，基于 TransferQueue + AgentLoopManager 的新架构）。
- use_v1=False → 走旧的 legacy/V0 trainer（main_ppo_v0.py 里的 TaskRunner），且已被标注废弃。

use_v1=True 时，TaskRunnerV1.run() 会做几件 V0 不做的事：
- 强制启用 TransferQueue（config.transfer_queue.enable = True），用它在 rollout / reward / actor 之间传数据。
- 按 trainer.v1.trainer_mode 选具体 trainer，支持三种模式（ppo_trainer.yaml 第 206 行）：
    - sync：同步 PPO（默认）。
    - colocate_async：trainer 与 rollout 同卡的异步流水（带 num_warmup_batches）。
    - separate_async：trainer 与 rollout 分离的异步，带 off-policy 控制（parameter_sync_step、max_off_policy_threshold 等）。
- 通过 AgentLoopManager 做 rollout（agent loop 抽象），并支持自定义 sampler / off-policy replay buffer。